In [3]:
from datetime import datetime, timedelta
import win32com.client
import pandas as pd
import pythoncom

# --- CONFIG ---
FILE_PATH = r'C:\Users\falve11887\OneDrive - Elekta\Documents\Monthly Clean up automations\Automations\Automation for monthly reports pt-2\Monthly_Installations_All_SW_All_Regions.xlsx'
VALID_VERSIONS_PATH = r"C:\Users\falve11887\OneDrive - Elekta\Bartalo, Renan's files - Project - Valid Version\Versions_Valid.xlsx"

CATEGORIES = {
    'ThinkQA / DOSIsoft': ['DOSISOFT SW EPIB', 'DOSISOFT SW EPIG', 'DOSISOFT SW MU2NET', 'THINKQA FOR LINAC', 'THINKQA FOR UNITY'],
    'AQUA': ['AQUA SOFTWARE'],
    'Monaco': ['MONACO SOFTWARE', 'MONACO'],
    'Monaco MIM': ['MIM CPAI SW', 'MIM MAESTRO SW'],
    'MOSAIQ': ['MOSAIQ Software'],
    'TELEPORT': ['MOSAIQ TELEPORT SOFTWARE'],
    'VOICE': ['MOSAIQ VOICE SOFTWARE'],
    'ASSIST': ['MOSAIQ ASSIST SOFTWARE'],
    'SMARTFLOW': ['SMART FLOW SOFTWARE', 'SRP - SMART FLOW 1.3 SW PRPTL', 'SRP - SMART FLOW SW SUBSCR'],
    'SMARTCLINIC': ['SMARTCLINIC SOFTWARE'],
    'KAIKU': ['KAIKU SOFTWARE'],
    'MOA': ['MOA Software']
}

# ==============================================================================
# ROUTINE: EXTRACT
# ==============================================================================

def run_excel_refresh(file_path):
    """
    Routine: EXTRACT
    Function: update_data
    Description: Triggers the Excel RefreshAll method via COM to update Salesforce 
    connections (OLEDB/PowerQuery) before processing.
    """
    print(">>> [REFRESH] Updating Salesforce connections in Excel...")
    pythoncom.CoInitialize()
    try:
        excel = win32com.client.Dispatch("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        wb = excel.Workbooks.Open(file_path)
        wb.RefreshAll()
        excel.CalculateUntilAsyncQueriesDone()
        wb.Save()
        wb.Close()
    finally:
        excel.Quit()
        pythoncom.CoUninitialize()

def get_raw_data(file_path):
    """
    Routine: EXTRACT
    Function: load_data
    Description: Loads the specific sheets from the Excel file and consolidates 
    them into a single DataFrame.
    """
    print(">>> [LOAD] Reading Excel sheets...")
    df_normal = pd.read_excel(file_path, sheet_name="All R except RCH - All SW's")
    df_rch = pd.read_excel(file_path, sheet_name="RCH - All SW's")
    return pd.concat([df_normal, df_rch], ignore_index=True)

# ==============================================================================
# ROUTINE: TRANSFORM
# ==============================================================================

def apply_version_validation(df, versions_path):
    """
    Routine: TRANSFORM
    Function: validate_versions
    Description: Cross-references product/version combinations against a master 
    validation file to flag missing or invalid entries.
    """
    print(">>> [VALIDATION] Checking software versions...")
    ref_df = pd.read_excel(versions_path)
    valid_set = set(f"{str(row['Product Sibex Name']).strip().lower()}|{str(row['Version Detail']).strip().lower()}" 
                   for _, row in ref_df.iterrows())
    
    def check(row):
        key = f"{str(row.get('Sibex Name', '')).strip().lower()}|{str(row.get('Version Detail', '')).strip().lower()}"
        ver = str(row.get('Version Detail', '')).lower()
        if ver in ['', 'nan']: return "(Missing SW Version)"
        return "" if key in valid_set else "(Invalid Version)"

    df['Validation_Note'] = df.apply(check, axis=1)
    return df

def categorize_products(df):
    """
    Routine: TRANSFORM
    Function: map_categories
    Description: Maps Sibex product names to high-level categories defined in 
    the CONFIG dictionary (Case Insensitive).
    """
    print(">>> [CATEGORIZE] Mapping products to categories...")
    product_map = {str(p).strip().lower(): cat for cat, prods in CATEGORIES.items() for p in prods}
    df['Sibex_Key'] = df['Sibex Name'].astype(str).str.strip().str.lower()
    df['Category'] = df['Sibex_Key'].map(product_map)
    return df

# ==============================================================================
# ROUTINE: COUNT
# ==============================================================================

def aggregate_statistics(df_reg, region_name):
    """
    Routine: COUNT
    Function: installations_count
    Description: Aggregates counts and details per region, handling special 
    business logic for 'Monaco MIM' and 'RCH' Monaco cross-referencing.
    """
    stats = {}
    is_rch = str(region_name).upper() == 'RCH'

    for cat in CATEGORIES.keys():
        cat_data = df_reg[df_reg['Category'] == cat].copy()
        
        if not cat_data.empty:
            num_sites = cat_data['Movex Account Number'].nunique()
            lines = []

            # SPECIAL CASE: Monaco in RCH (Cross-referencing "Monaco" with "Monaco Software" versions)
            if cat == 'Monaco' and is_rch:
                version_fix_map = cat_data[cat_data['Sibex Name'].str.strip().str.upper() == 'MONACO SOFTWARE'].set_index('Movex Account Number')['Validation_Note'].to_dict()
                monaco_counts = cat_data[cat_data['Sibex Name'].str.strip().str.upper() == 'MONACO']
                
                summary = monaco_counts.groupby(['Movex Account Number', 'Account: Account Name']).agg({'Quantity': 'sum'}).reset_index()
                for _, row in summary.iterrows():
                    acc_num = row['Movex Account Number']
                    note = version_fix_map.get(acc_num, "")
                    line = f"{int(row['Quantity'])}x for {acc_num} Account: {row['Account: Account Name']} {note}".strip()
                    lines.append(line)

            # SPECIAL CASE: Monaco MIM (Splitting CPAI & MAESTRO)
            elif cat == 'Monaco MIM':
                accounts = cat_data.groupby(['Movex Account Number', 'Account: Account Name', 'Validation_Note'])
                for (m_acc, acc_name, val_note), group in accounts:
                    cpai_qty = int(group[group['Sibex Name'].str.contains('CPAI', case=False, na=False)]['Quantity'].sum())
                    maestro_qty = int(group[group['Sibex Name'].str.contains('MAESTRO', case=False, na=False)]['Quantity'].sum())
                    parts = []
                    if maestro_qty > 0: parts.append(f"{maestro_qty}x MAESTRO")
                    if cpai_qty > 0: parts.append(f"{cpai_qty}x CPAI")
                    sub_detail = " & ".join(parts)
                    line = f"{sub_detail} for {m_acc} Account: {acc_name} {val_note}".strip()
                    lines.append(line)

            # DEFAULT LOGIC
            else:
                summary = cat_data.groupby(['Movex Account Number', 'Account: Account Name', 'Validation_Note']).agg({'Quantity': 'sum'}).reset_index()
                for _, row in summary.iterrows():
                    line = f"{int(row['Quantity'])}x for {row['Movex Account Number']} Account: {row['Account: Account Name']} {row['Validation_Note']}".strip()
                    lines.append(line)
            
            stats[cat] = {'sites': num_sites, 'lines': lines}
        else:
            stats[cat] = None
    return stats

# ==============================================================================
# ROUTINE: REPORTING
# ==============================================================================

def build_report_text(stats, region_name):
    """
    Routine: REPORTING
    Function: format_report
    Description: Generates the final formatted text block for a specific region, 
    including standard notes and conditional error warnings.
    """
    last_month = datetime.now().replace(day=1) - timedelta(days=1)
    has_validation_issue = False
    category_blocks = []
    missing_cats = []

    for cat, data in stats.items():
        if data:
            category_blocks.append(f"{cat}: {data['sites']} sites")
            for line in data['lines']:
                category_blocks.append(f"    {line}")
                if "(Missing SW Version)" in line or "(Invalid Version)" in line:
                    has_validation_issue = True
        else:
            missing_cats.append(cat)

    report = [
        "=" * 80,
        f"{region_name} | {last_month.strftime('%B %Y')} – Software Installations & Data Check",
        "=" * 80 + "\n",
        "Hello Team,",
        f"Here's a quick update from the {last_month.strftime('%B')} CLM extract:\n",
        "Note: If you notice anything in this report that did not happen or installations that haven't been registered in CLM,",
        "please let us know so we can adjust accordingly to keep everything accurate."
    ]

    if has_validation_issue:
        report.append("Important: Please make sure to insert the missing software version correctly where indicated.")
    
    report.append("")
    report.extend(category_blocks)

    if missing_cats:
        report.append(f"\n{', '.join(missing_cats)}: No installations Recorded")

    report.extend([f"\n{region_name} All SW's | Salesforce", "Best regards,", "Fernando", "\n" + "-"*80 + "\n"])
    return "\n".join(report)

# ==============================================================================
# ROUTINE: ORCHESTRATION
# ==============================================================================

def main():
    """
    Routine: ORCHESTRATION
    Function: execute_pipeline
    Description: Orchestrates the full process flow from extraction to file saving.
    """
    # 1. Extraction
    run_excel_refresh(FILE_PATH)
    df = get_raw_data(FILE_PATH)
    
    # 2. Transformation
    df = categorize_products(df)
    df = apply_version_validation(df, VALID_VERSIONS_PATH)
    
    # 3. Processing & Formatting
    all_reports = []
    regions = sorted(df['New Model Account Region'].dropna().unique())
    
    for reg in regions:
        print(f">>> [MAIN] Processing region: {reg}")
        df_reg = df[df['New Model Account Region'] == reg].copy()
        
        stats = aggregate_statistics(df_reg, reg)
        all_reports.append(build_report_text(stats, reg))
    
    # 4. Save to File (Using Last Month in Filename)
    last_month = datetime.now().replace(day=1) - timedelta(days=1)
    file_output = f"Full_Report_{last_month.strftime('%B_%Y')}.txt"
    
    with open(file_output, "w", encoding="utf-8") as f:
        f.write("\n".join(all_reports))
    
    print(f"\n>>> [SUCCESS] {last_month.strftime('%B %Y')} report generated: {file_output}")

if __name__ == "__main__":
    main()

>>> [REFRESH] Updating Salesforce connections in Excel...
>>> [LOAD] Reading Excel sheets...
>>> [CATEGORIZE] Mapping products to categories...
>>> [VALIDATION] Checking software versions...
>>> [MAIN] Processing region: RCH
>>> [MAIN] Processing region: REU
>>> [MAIN] Processing region: RJP
>>> [MAIN] Processing region: RNCA
>>> [MAIN] Processing region: RSA
>>> [MAIN] Processing region: RTIMEA

>>> [SUCCESS] May 2026 report generated: Full_Report_May_2026.txt
